<a href="https://colab.research.google.com/github/aditya-dwivedi13/RL-TicTacToe/blob/main/Tic_Tac_Toe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import sys
sys.path.append('/content/drive/MyDrive/')

In [3]:
from tictactoe import Game,Board,Player
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import time

In [4]:
class CNNModel(nn.Module):
    def __init__(self,load=False,name=None):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=2,
            out_channels=16,
            kernel_size=2)
        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, 1)
        if load and name is not None:
           self.load(name)

    def forward(self,state,move):
      x = torch.stack((state,move),dim=0)
      x = x.unsqueeze(0)
      x = F.relu(self.conv1(x))
      x = torch.flatten(x,1)
      x = F.relu(self.fc1(x))
      score = torch.tanh(self.fc2(x))
      return score

    def save(self, name="cnn_model.pth"):
       folder = "/content/drive/MyDrive/models"
       os.makedirs(folder, exist_ok=True)
       path = os.path.join(folder, name)
       torch.save(self.state_dict(), path)
       print(f"Model saved to: {path}")

    def load(self, name):
       folder = "/content/drive/MyDrive/models"
       path = os.path.join(folder, name)
       self.load_state_dict(torch.load(path))
       self.eval()
       print(f"Model loaded from: {path}")

In [5]:
class AIAgent:

    def __init__(self, model):
        self.model = model

    def encode_move(self, block,value):
      block = block-0.001
      x = int(block//3)
      y = round(block%3) - 1
      position = (x,y)
      move = torch.zeros((3,3), dtype=torch.float32)
      x, y = position
      move[x, y] = value
      return move

    def encode_state(self, board):
      return torch.tensor(board.state,dtype=torch.float32)

    def legal_blocks(self, states):
      # states should be a flatten numpy array
      blocks = [i+1 for i in range(9) if states[i] == 0]
      return blocks

    def choose_move(self, board, encode=True, typ=1):

      state = self.encode_state(board)
      best_score = -float("inf")
      best_move = None

      for block in self.legal_blocks(board.state.flatten()):
          encoded = self.encode_move(block,value=typ)
          score = self.model(state, encoded)
          score = score.item()

          if score > best_score:
              best_score = score
              if encode:
                best_move= encoded
              else:
                best_move = block

      return best_move

In [6]:
class AIGame:
  def __init__(self,model) -> None:
     self.game = Game()
     self.model = model
     self.agent = AIAgent(self.model)

  def human_move(self):
    block = int(input('Your move enter a block number: '))
    return block

  def ai_move(self,value):
    position = self.agent.choose_move(board = self.game.board, encode=False,typ=value)
    return position

  def play_(self,human=1):
    self.game.board.display()
    turn = 1
    value = -1 if human == 1 else 1
    while True:
      if turn==2:
        player = self.game.player2
      else:
        player = self.game.player1

      if human == turn:
        block = self.human_move()
      else:
        block = self.ai_move(value)
        print('Thinking..')
        time.sleep(2)
        print('AI Move',block)

      cmd = self.game.move(player,block)
      if cmd == "play":
        self.game.board.display()
        turn = 3-turn
      else:
        continue

      r = self.game.checkwin()
      if r==0:
        if self.game.checkdraw():
          print("Draw")
          break
      else:
        print(f"Player{r} Won!!")
        break

In [7]:
model = CNNModel(load=False)

In [8]:
game = AIGame(model)
game.play_()

_ _ _
_ _ _
_ _ _
Your move enter a block number: 2
_ O _
_ _ _
_ _ _
Thinking..
AI Move 9
_ O _
_ _ _
_ _ X
Your move enter a block number: 3
_ O O
_ _ _
_ _ X
Thinking..
AI Move 6
_ O O
_ _ X
_ _ X
Your move enter a block number: 1
O O O
_ _ X
_ _ X
Player1 Won!!
